# InsightAid AI — Sentiment Model Training (Objective 1)

Trains and evaluates all three sentiment models (XLM-R, mBERT, TF-IDF +
Logistic Regression) against the real `insightaid_sentiment` dataset
prepared by `01_sentiment_data_preparation.ipynb`
(`dataset_version` recorded in `manifest.json`), across both evaluation
paths (original-language, translate-to-English), using the already-tested
code in `ml/sentiment/src/` — no training logic is duplicated here.

**Before running:** Runtime → Change runtime type → **T4 GPU**. XLM-R/mBERT
will be extremely slow on CPU.

**What this does NOT do:** invent any metric. Every number below comes from
`ml/sentiment/src/metrics.py` run against this model's real predictions on
the real held-out test set. If something fails partway, the cells already
run still produced real results — nothing here back-fills a guess.

In [3]:
import os, sys, zipfile

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Google Drive mount failed. Proceeding without Drive mounting.")
        print("Error:", e)

# Fallback dataset path logic if Drive is unavailable
RAW_DATA_PATH = "/content/drive/MyDrive/INSIGHTAID_AI/ml_data/insightaid_sentiment_data.zip"

# If Drive didn't mount, check inside the cloned repository subdirectory or local folder
if not os.path.exists(RAW_DATA_PATH):
    local_fallback = "/content/insightaid-ai/ml/sentiment/data/insightaid_sentiment_data.zip"
    # Create a dummy folder structure if missing, or notify the user to upload
    if os.path.exists(local_fallback):
        RAW_DATA_PATH = local_fallback
    else:
        print("⚠️ Please upload your dataset zip file ('insightaid_sentiment_data.zip') directly to Colab or configure Drive.")
        # Let's create a placeholder/fallback path for user upload
        RAW_DATA_PATH = "/content/insightaid_sentiment_data.zip"

if os.path.exists(RAW_DATA_PATH):
    if RAW_DATA_PATH.endswith(".zip"):
        DATA_DIR = "/content/insightaid_sentiment_data"
        print(f"Extracting {RAW_DATA_PATH} to {DATA_DIR}...")
        with zipfile.ZipFile(RAW_DATA_PATH, 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
    else:
        DATA_DIR = RAW_DATA_PATH
    print("Dataset folder:", DATA_DIR)
    print(os.listdir(DATA_DIR))
else:
    print(f"[Expected Path] Dataset path not found: {RAW_DATA_PATH}. Please upload the file manually to /content/.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting /content/drive/MyDrive/INSIGHTAID_AI/ml_data/insightaid_sentiment_data.zip to /content/insightaid_sentiment_data...
Dataset folder: /content/insightaid_sentiment_data
['test_by_language', 'label_mapping.json', 'all_harmonised.parquet', 'DATASET_REPORT.md', 'validation.csv', 'test.csv', 'manifest.json', 'train.csv']


In [11]:
import os, sys, zipfile

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True)
    except Exception as e:
        print("Google Drive mount failed. Proceeding without Drive mounting.")
        print("Error:", e)

RAW_DATA_PATH = "/content/drive/MyDrive/INSIGHTAID_AI/ml_data/insightaid_sentiment_data.zip"

if not os.path.exists(RAW_DATA_PATH):
    local_fallback = "/content/insightaid-ai/ml/sentiment/data/insightaid_sentiment_data.zip"
    if os.path.exists(local_fallback):
        RAW_DATA_PATH = local_fallback
    else:
        print("⚠️ Please upload your dataset zip file ('insightaid_sentiment_data.zip') directly to Colab.")
        RAW_DATA_PATH = "/content/insightaid_sentiment_data.zip"

if os.path.exists(RAW_DATA_PATH):
    if RAW_DATA_PATH.endswith(".zip"):
        DATA_DIR = "/content/insightaid_sentiment_data"
        print(f"Extracting {RAW_DATA_PATH} to {DATA_DIR}...")
        with zipfile.ZipFile(RAW_DATA_PATH, 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
    else:
        DATA_DIR = RAW_DATA_PATH
    print("Dataset folder:", DATA_DIR)
    print(os.listdir(DATA_DIR))
else:
    print(f"[Expected Path] Dataset path not found: {RAW_DATA_PATH}.")

Mounted at /content/drive
Extracting /content/drive/MyDrive/INSIGHTAID_AI/ml_data/insightaid_sentiment_data.zip to /content/insightaid_sentiment_data...
Dataset folder: /content/insightaid_sentiment_data
['DATASET_REPORT.md', 'manifest.json', 'test.csv', 'label_mapping.json', 'test_by_language', 'validation.csv', 'all_harmonised.parquet', 'train.csv']


In [4]:
# 2. Get ml/sentiment/src (clone the repo — it's public on GitHub).
REPO_DIR = "/content/insightaid-ai"
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://github.com/danishalif04/InsightAID-AI.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

sys.path.insert(0, os.path.join(REPO_DIR, "ml", "sentiment"))
print("ml/sentiment/src is now importable.")

Cloning into '/content/insightaid-ai'...
remote: Enumerating objects: 312, done.
remote: Counting objects: 100% (312/312), done.
remote: Compressing objects: 100% (287/287), done.
remote: Total 312 (delta 11), reused 276 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (312/312), 425.05 KiB | 10.63 MiB/s, done.
Resolving deltas: 100% (11/11), done.
ml/sentiment/src is now importable.


In [5]:
# 3. Install training dependencies (pandas/numpy/scikit-learn are already
# in Colab's base image — only the heavy transformer stack is installed here).
!pip install -q torch transformers datasets accelerate sentencepiece sacremoses huggingface_hub

import torch
print("GPU available:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (fix this before training XLM-R/mBERT)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 21.4 MB/s eta 0:00:00
GPU available: True - Tesla T4


In [6]:
# 4. Load the real prepared dataset and confirm which dataset_version this run uses.
import json as _json
import pandas as pd

with open(os.path.join(DATA_DIR, "manifest.json")) as f:
    manifest = _json.load(f)
DATASET_VERSION = manifest["dataset_version"]
print("dataset_version:", DATASET_VERSION, "| rows:", manifest["rows"])

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "validation.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

# The prepared files use 'text'/'label' columns (src/sentiment_prep.py's
# STD_COLS) — NOT 'cleaned_text', which is what the app's own processed_data
# export (src/services/mlExportService.js) uses instead. Both
# train_xlmr.train_transformer() and evaluate.run_full_comparison() accept
# text_column/label_column as parameters for exactly this reason.
TEXT_COL, LABEL_COL = "text", "label"
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
assert TEXT_COL in train_df.columns and LABEL_COL in train_df.columns

dataset_version: v20261012-ba90cb55d8f7 | rows: 107171
train=85639  val=10763  test=10769


In [12]:
import os
import pandas as pd
import json as _json

DATA_DIR = "/content/insightaid_sentiment_data"

with open(os.path.join(DATA_DIR, "manifest.json")) as f:
    manifest = _json.load(f)
DATASET_VERSION = manifest["dataset_version"]
print("dataset_version:", DATASET_VERSION, "| rows:", manifest["rows"])

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "validation.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

TEXT_COL, LABEL_COL = "text", "label"
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

dataset_version: v20261012-ba90cb55d8f7 | rows: 107171
train=85639  val=10763  test=10769


In [7]:
# 5. Train the TF-IDF + Logistic Regression baseline (fast, CPU-only —
# a good sanity check before committing time to the transformer runs).
from src.train_tfidf_baseline import train_tfidf_baseline, predict as tfidf_predict

tfidf_model = train_tfidf_baseline(train_df[TEXT_COL], train_df[LABEL_COL])
print("TF-IDF baseline trained on", len(train_df), "rows.")

TF-IDF baseline trained on 85639 rows.


In [14]:
from src.train_tfidf_baseline import train_tfidf_baseline, predict as tfidf_predict

# Train the baseline TF-IDF model
tfidf_model = train_tfidf_baseline(train_df[TEXT_COL], train_df[LABEL_COL])
print("TF-IDF baseline successfully trained on", len(train_df), "rows!")

TF-IDF baseline successfully trained on 85639 rows!


In [8]:
# 6. Train XLM-RoBERTa (primary model, FYP2.md §46). Needs the GPU runtime.
from pathlib import Path
from src.train_xlmr import XLMR_CHECKPOINT, train_transformer, predict as xlmr_predict

xlmr_trained = train_transformer(
    XLMR_CHECKPOINT, train_df, val_df,
    output_dir=Path("/content/xlmr_model"),
    text_column=TEXT_COL, label_column=LABEL_COL,
)
print("XLM-R trained. Saved to", xlmr_trained.output_dir)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

XLM-R trained. Saved to /content/xlmr_model


In [ ]:
# 7. Train mBERT (comparison model — this project's own methodology
# addition, not required by FYP2.md '46).
import os
import sys
import importlib
from pathlib import Path
import pandas as pd

REPO_DIR = "/content/insightaid-ai"
DATA_DIR = "/content/insightaid_sentiment_data"

# Automatically re-clone the repository if it disappeared due to a session reset
if not os.path.exists(REPO_DIR):
    print(f"'{REPO_DIR}' not found. Cloning repository...")
    !git clone --depth 1 https://github.com/danishalif04/InsightAID-AI.git {REPO_DIR}

# Ensure datasets are loaded into memory
if 'train_df' not in globals() or 'val_df' not in globals():
    print("Loading datasets...")
    train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
    val_df = pd.read_csv(os.path.join(DATA_DIR, "validation.csv"))

if 'TEXT_COL' not in globals() or 'LABEL_COL' not in globals():
    TEXT_COL, LABEL_COL = "text", "label"

_sentiment_path = ""

# Let's search recursively for where 'src/train_mbert.py' exists
for root, dirs, files in os.walk(REPO_DIR):
    if "train_mbert.py" in files and "src" in root:
        # We want the parent of 'src'
        _sentiment_path = os.path.abspath(os.path.dirname(root))
        break

print("Dynamically found sentiment root path:", _sentiment_path)

# Ensure it is at the very front of sys.path
if _sentiment_path and _sentiment_path not in sys.path:
    sys.path.insert(0, _sentiment_path)

# Invalidate import caches to force Python to recognize the newly added path
importlib.invalidate_caches()

from src.train_mbert import train_mbert
from src.train_xlmr import predict as mbert_predict  # same generic predict()

mbert_trained = train_mbert(
    train_df,
    val_df,
    output_dir=Path("/content/mbert_model"),
    text_column=TEXT_COL,
    label_column=LABEL_COL,
)
print("mBERT trained. Saved to", mbert_trained.output_dir)

Loading datasets...
Dynamically found sentiment root path: /content/insightaid-ai/ml/sentiment


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/

Epoch,Training Loss,Validation Loss


In [ ]:
# 8. Evaluate all three models across both evaluation paths (original-
# language vs. translate-to-English first) on the held-out test set.
from src import evaluate as evaluate_module

RUN_VERSION = pd.Timestamp.utcnow().strftime("v%Y%m%d-%H%M%S")  # this training run's own model_version

predict_fns = {
    "tfidf_lr": lambda texts: tfidf_predict(tfidf_model, texts),
    "xlmr": lambda texts: xlmr_predict(xlmr_trained, texts),
    "mbert": lambda texts: mbert_predict(mbert_trained, texts),
}

rows = evaluate_module.run_full_comparison(
    predict_fns, test_df,
    model_version=RUN_VERSION,
    dataset_version=DATASET_VERSION,
    training_samples=len(train_df),
    validation_samples=len(val_df),
    text_column=TEXT_COL, label_column=LABEL_COL,
    results_dir=Path("/content/results"),
)

for row in rows:
    print(f"{row.model_type:10s} {row.eval_path:22s} acc={row.accuracy:.4f} f1={row.f1:.4f} n={row.test_samples}")

In [13]:
import os
import pandas as pd
from pathlib import Path
from src import evaluate as evaluate_module

# Ensure RUN_VERSION is defined
RUN_VERSION = pd.Timestamp.utcnow().strftime("v%Y%m%d-%H%M%S")

# Construct prediction wrappers based on what is available
predict_fns = {}

# 1. TF-IDF baseline prediction
if 'tfidf_model' in globals():
    from src.train_tfidf_baseline import predict as tfidf_predict
    predict_fns["tfidf_lr"] = lambda texts: tfidf_predict(tfidf_model, texts)
    print("✅ Added TF-IDF baseline to evaluation list.")

# 2. XLM-RoBERTa prediction
if 'xlmr_trained' in globals():
    from src.train_xlmr import predict as xlmr_predict
    predict_fns["xlmr"] = lambda texts: xlmr_predict(xlmr_trained, texts)
    print("✅ Added XLM-R to evaluation list.")

# 3. mBERT prediction
if 'mbert_trained' in globals():
    from src.train_xlmr import predict as mbert_predict
    predict_fns["mbert"] = lambda texts: mbert_predict(mbert_trained, texts)
    print("✅ Added mBERT to evaluation list.")

if not predict_fns:
    print("⚠️ No models have been trained in this session yet. Please train at least one model before evaluation.")
else:
    print(f"Evaluating {list(predict_fns.keys())} on test dataset (n={len(test_df)})...\n")

    rows = evaluate_module.run_full_comparison(
        predict_fns, test_df,
        model_version=RUN_VERSION,
        dataset_version=DATASET_VERSION,
        training_samples=len(train_df),
        validation_samples=len(val_df),
        text_column=TEXT_COL, label_column=LABEL_COL,
        results_dir=Path("/content/results"),
    )

    print("\nEvaluation Results:")
    for row in rows:
        print(f"{row.model_type:10s} {row.eval_path:22s} acc={row.accuracy:.4f} f1={row.f1:.4f} n={row.test_samples}")

⚠️ No models have been trained in this session yet. Please train at least one model before evaluation.


## Optional: evaluate against the hand-labelled gold test set

Only run this once `gold_agreed.csv` exists (built from
`gold_test_set/ANNOTATION_GUIDELINES.md`'s double-annotation + agreement
process — a separate, parallel task, not part of this notebook). Per the
guidelines, this is reported **separately** from the public-dataset test
split above, never blended into it — it's the only part of the evaluation
that covers real Malaysian code-mixed text.

In [ ]:
GOLD_PATH = os.path.join(DATA_DIR, "..", "gold_test_set", "gold_agreed.csv")
if os.path.exists(GOLD_PATH):
    gold_df = pd.read_csv(GOLD_PATH)
    gold_df = gold_df[gold_df["final_label"].isin(["positive", "neutral", "negative"])]  # drop 'exclude' rows
    print("Gold test set:", len(gold_df), "items")

    gold_rows = []
    for model_type, predict_fn in predict_fns.items():
        y_pred = predict_fn(list(gold_df["text"]))
        computed, row = evaluate_module.evaluate_predictions(
            list(gold_df["final_label"]), y_pred, list(gold_df["language"]),
            model_name=f"insightaid-sentiment-{model_type}",
            model_version=f"{RUN_VERSION}-gold",  # distinct from the public test-set run (unique constraint on model_versions)
            model_type=model_type,
            eval_path="original_language",  # gold set has no translate-first variant
            dataset_version=DATASET_VERSION,
            training_samples=len(train_df),
            validation_samples=len(val_df),
            test_samples=len(gold_df),
            notes="Evaluated against the hand-labelled gold_test_set, not the public dataset's own test split.",
        )
        print(f"{model_type:10s} gold_test_set  acc={row.accuracy:.4f} f1={row.f1:.4f} n={row.test_samples}")
        gold_rows.append(row)
    rows = rows + gold_rows
else:
    print("No gold_agreed.csv yet at", GOLD_PATH, "— skipping. Build it per gold_test_set/ANNOTATION_GUIDELINES.md first.")

## Record these results in Supabase

Nothing in this app writes to `model_versions` automatically (by design —
only the Supabase **service role** can insert; see
`supabase/migrations/0004_model_versions.sql`). Copy the SQL printed below
into Supabase Dashboard → SQL Editor → New query → Run, once per row.

In [ ]:
def sql_literal(value):
    if value is None:
        return "NULL"
    if isinstance(value, (dict, list)):
        return "'" + _json.dumps(value).replace("'", "''") + "'::jsonb"
    if isinstance(value, str):
        return "'" + value.replace("'", "''") + "'"
    return str(value)

from dataclasses import asdict

for row in rows:
    d = asdict(row)
    cols = ", ".join(d.keys())
    vals = ", ".join(sql_literal(v) for v in d.values())
    print(f"INSERT INTO model_versions ({cols}) VALUES ({vals});")
    print()

## Save everything back to Drive

So the trained models and metrics JSON survive after this Colab session ends.

In [ ]:
import shutil

RESULTS_DEST = os.path.join(DATA_DIR, "..", "training_runs", RUN_VERSION)
os.makedirs(RESULTS_DEST, exist_ok=True)
if os.path.exists("/content/results"):
    shutil.copytree("/content/results", os.path.join(RESULTS_DEST, "metrics"), dirs_exist_ok=True)
print("Metrics JSON copied to", RESULTS_DEST)
print("(The model checkpoints themselves are large — /content/xlmr_model and /content/mbert_model —")
print(" copy those to Drive too with shutil.copytree if you intend to serve this exact model later.)")